In [ ]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

# Нейронная сеть: от основ.


Эта глава знаменует начало пути, в ходе которого мы глубоко изучим внутреннюю структуру моделей, которые мы использовали в предыдущих главах. Мы рассмотрим многие из тех же тем, что и раньше, но в этот раз мы сосредоточимся гораздо больше на деталях реализации и гораздо меньше на практических вопросах того, как и почему все устроено именно так.

Мы будем создавать все с нуля, используя только базовые операции индексации в тензорах. Мы напишем нейронную сеть с самого начала, а затем реализуем обратное распространение ошибки вручную, чтобы точно понимать, что происходит в PyTorch, когда мы вызываем `loss.backward`. Мы также увидим, как расширить PyTorch с помощью пользовательских функций *autograd*, которые позволяют нам определять собственные операции прямого и обратного распространения.

## Создание слоя нейронной сети с нуля.

Начнем с того, что освежим наши знания о том, как используется умножение матриц в базовой нейронной сети. Поскольку мы строим все с нуля, изначально мы будем использовать только чистый Python (за исключением обращения к тензорам PyTorch), а затем заменим этот чистый Python функциональностью PyTorch, когда мы поймем, как ее создать.

### Моделирование нейрона


Нейрон получает определенное количество входных сигналов и имеет внутренний вес для каждого из них. Он суммирует эти взвешенные входные сигналы для получения выходного значения и добавляет внутренний смещение. В математической записи это можно представить следующим образом:

$$ out = \sum_{i=1}^{n} x_{i} w_{i} + b$$

где $x_i$ - входные сигналы, $w_i$ - веса, а $b$ - смещение. В коде это выглядит так:

```python
output = sum([x*w for x,w in zip(inputs,weights)]) + bias
```

Этот выходной сигнал затем передается нелинейной функции, называемой *функцией активации*, перед тем, как быть отправленным в другой нейрон. В глубоком обучении наиболее распространенной функцией активации является *выпрямленная линейная функция*, или *ReLU*, которая, как мы уже видели, является элегантным способом записи следующего:
```python
def relu(x): return x if x >= 0 else 0
```

Для построения модели глубокого обучения нейроны объединяются в последовательные слои. Мы создаем первый слой, состоящий из определенного количества нейронов (известного как *размер скрытого слоя*), и соединяем все входные данные с каждым из этих нейронов. Такой слой часто называют *полносвязным слоем* или *плотным слоем* (из-за плотной связи), или *линейным слоем*.

Для каждого `входного значения` в нашей выборке и для каждого нейрона с заданным `весом` необходимо вычислить скалярное произведение:

```python
sum([x*w for x,w in zip(input,weight)])
```

Если вы немного знакомы с линейной алгеброй, вы, возможно, помните, что большое количество таких скалярных произведений возникает при *умножении матриц*. Более точно, если наши входные данные представлены в матрице `x` размером `batch_size` x `n_inputs`, а веса наших нейронов сгруппированы в матрице `w` размером `n_neurons` x `n_inputs` (каждый нейрон должен иметь такое же количество весов, как и входных данных), и все смещения объединены в векторе `b` размером `n_neurons`, то выход этого полносвязного слоя будет следующим:

```python
y = x @ w.t() + b
```

где `@` обозначает матричное произведение, а `w.t()` – транспонированная матрица `w`. Выход `y` имеет размер `batch_size` x `n_neurons`, и в позиции `(i,j)` (для тех, кто любит математику):

$$y_{i,j} = \sum_{k=1}^{n} x_{i,k} w_{k,j} + b_{j}$$

Или в коде:

```python
y[i,j] = sum([a * b for a,b in zip(x[i,:],w[j,:])]) + b[j]
```

Транспонирование необходимо, поскольку в математическом определении матричного произведения `m @ n`, коэффициент `(i,j)` вычисляется следующим образом:

```python
sum([a * b for a,b in zip(m[i,:],n[:,j])])
```

Таким образом, основная операция, которая нам нужна, – это умножение матриц, поскольку именно она лежит в основе работы нейронной сети.


### Умножение матриц с нуля.


Давайте напишем функцию, которая вычисляет произведение двух тензоров, прежде чем мы начнем использовать версию этой функции, реализованную в PyTorch. Мы будем использовать только индексацию тензоров PyTorch:


In [ ]:
import torch
from torch import tensor

Нам потребуется три вложенных цикла `for`: один для индексов строк, один для индексов столбцов и один для внутреннего суммирования. `ac` и `ar` обозначают количество столбцов матрицы `a` и количество строк матрицы `a` соответственно (такая же соглашение используется для матрицы `b`), и мы убедимся, что вычисление произведения матриц возможно, проверив, что у матрицы `a` столько же столбцов, сколько у матрицы `b` строк:

In [ ]:
def matmul(a,b):
    ar,ac = a.shape # n_rows * n_cols
    br,bc = b.shape
    assert ac==br
    c = torch.zeros(ar, bc)
    for i in range(ar):
        for j in range(bc):
            for k in range(ac): c[i,j] += a[i,k] * b[k,j]
    return c

Для демонстрации этого подхода, давайте представим (используя случайные матрицы), что мы работаем с небольшой выборкой из 5 изображений MNIST, преобразованных в векторы размером 28x28, и используем линейную модель для преобразования этих векторов в 10 активаций:


In [ ]:
m1 = torch.randn(5,28*28)
m2 = torch.randn(784,10)

Давайте измерим время выполнения нашей функции, используя специальную команду Jupyter `%time`:


In [ ]:
%time t1=matmul(m1, m2)

CPU times: user 1.15 s, sys: 4.09 ms, total: 1.15 s
Wall time: 1.15 s


И посмотрите, как это соотносится с встроенным оператором `@` в PyTorch:

In [ ]:
%timeit -n 20 t2=m1@m2

14 µs ± 8.95 µs per loop (mean ± std. dev. of 7 runs, 20 loops each)


Как мы видим, использование трех вложенных циклов в Python – это очень плохая идея! Python – это язык с относительно низкой производительностью, и такой подход будет крайне неэффективным. Мы видим, что PyTorch работает примерно в 100 000 раз быстрее, чем Python, и это еще до того, как мы начнем использовать графический процессор (GPU)!

Откуда берется такая разница? PyTorch не реализовал умножение матриц на Python, а на C++, чтобы обеспечить высокую скорость. В целом, при выполнении вычислений над тензорами необходимо *векторизовать* их, чтобы воспользоваться преимуществами скорости PyTorch. Обычно для этого используются два подхода: поэлементные арифметические операции и механизм "broadcasting".

### Арифметические операции над элементами массива


Все основные операторы (`+`, `-`, `*`, `/`, `>`, `<`, `==`) могут применяться поэлементно. Это означает, что если мы записываем `a + b` для двух тензоров `a` и `b`, имеющих одинаковую форму, мы получим тензор, состоящий из сумм соответствующих элементов тензоров `a` и `b`:


In [ ]:
a = tensor([10., 6, -4])
b = tensor([2., 8, 7])
a + b

tensor([12., 14.,  3.])

Операторы логического типа (Boolean) возвращают массив значений типа "логический":


In [ ]:
a < b

tensor([False,  True,  True])

Если мы хотим узнать, является ли каждый элемент массива `a` меньше, чем соответствующий элемент в массиве `b`, или равны ли два тензора, нам необходимо объединить эти операции, выполняемые поэлементно, с функцией `torch.all`:

In [ ]:
(a < b).all(), (a==b).all()

(tensor(False), tensor(False))

Операции, такие как `all()`, `sum()` и `mean()`, возвращают тензоры, содержащие только один элемент, которые называются тензорами нулевого ранга. Если вам нужно преобразовать этот тензор в обычный объект Python (логическое значение или число), необходимо вызвать метод `.item()`:


In [ ]:
(a + b).mean().item()

9.666666984558105

Операции, выполняемые поэлементно, работают с тензорами любой размерности, при условии, что они имеют одинаковую форму:


In [ ]:
m = tensor([[1., 2, 3], [4,5,6], [7,8,9]])
m*m

tensor([[ 1.,  4.,  9.],
        [16., 25., 36.],
        [49., 64., 81.]])

Однако, вы не можете выполнять операции над тензорами поэлементно, если они не имеют одинаковую форму (если только они не являются "расширяемыми", как будет описано в следующем разделе).

In [ ]:
n = tensor([[1., 2, 3], [4,5,6]])
m*n

RuntimeError: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 0

Используя поэлементные арифметические операции, мы можем убрать один из трех вложенных циклов: мы можем умножить тензоры, соответствующие `i`-й строке матрицы `a` и `j`-му столбцу матрицы `b`, перед суммированием всех элементов. Это ускорит процесс, поскольку внутренний цикл теперь будет выполняться PyTorch на скорости, близкой к скорости выполнения кода на языке C.

Чтобы получить доступ к одному столбцу или строке, мы можем просто написать `a[i,:]` или `b[:,j]`. Символ `:` означает выбор всех элементов в этом измерении. Мы можем ограничить это и выбрать только часть указанного измерения, указав диапазон, например `1:5`, вместо просто `:`. В этом случае мы выберем элементы из столбцов или строк с 1 по 4 (второе число не включается).

Одно упрощение заключается в том, что мы всегда можем опускать завершающий двоеточие, поэтому `a[i,:]` можно сократить до `a[i]`. Учитывая все это, мы можем написать новую версию нашей операции умножения матриц:


In [ ]:
def matmul(a,b):
    ar,ac = a.shape
    br,bc = b.shape
    assert ac==br
    c = torch.zeros(ar, bc)
    for i in range(ar):
        for j in range(bc): c[i,j] = (a[i] * b[:,j]).sum()
    return c

In [ ]:
%timeit -n 20 t3 = matmul(m1,m2)

1.7 ms ± 88.1 µs per loop (mean ± std. dev. of 7 runs, 20 loops each)


Мы уже примерно в 700 раз быстрее, просто удалив этот внутренний цикл `for`! И это только начало – с использованием механизма вещания (broadcasting) мы можем убрать еще один цикл и добиться еще более значительного увеличения скорости.

### Вещание


Как мы обсуждали в разделе <<chapter_mnist_basics>>, "broadcasting" – это термин, введенный библиотекой [NumPy](https://docs.scipy.org/), который описывает, как тензоры разной размерности обрабатываются во время арифметических операций. Например, очевидно, что нельзя сложить матрицу 3x3 с матрицей 4x5, но что, если мы хотим сложить скаляр (который можно представить как тензор 1x1) с матрицей? Или вектор размера 3 с матрицей 3x4? В обоих случаях мы можем найти способ, чтобы эта операция имела смысл.

Broadcasting определяет конкретные правила, которые позволяют определить, совместимы ли формы тензоров при выполнении операций над элементами, и как тензор меньшей размерности расширяется, чтобы соответствовать тензору большей размерности. Важно освоить эти правила, если вы хотите писать код, который работает быстро. В этом разделе мы расширим наше предыдущее описание broadcasting, чтобы лучше понять эти правила.

#### Трансляция с использованием скалярного значения.

Трансляция с использованием скалярного значения – это самый простой тип трансляции. Когда у нас есть тензор `a` и скаляр, мы представляем себе тензор той же формы, что и `a`, заполненный этим скалярным значением, и выполняем операцию:


In [ ]:
a = tensor([10., 6, -4])
a > 0

tensor([ True,  True, False])

Как мы можем выполнить это сравнение? Значение `0` *транслируется* (broadcast) так, чтобы оно имело те же размеры, что и `a`. Обратите внимание, что это делается без создания тензора, заполненного нулями, в памяти (это было бы очень неэффективно).

Это очень полезно, если вы хотите нормализовать свой набор данных, вычитая среднее значение (скаляр) из всего набора данных (матрицы) и деля результат на стандартное отклонение (еще один скаляр):

In [ ]:
m = tensor([[1., 2, 3], [4,5,6], [7,8,9]])
(m - 5) / 2.73

tensor([[-1.4652, -1.0989, -0.7326],
        [-0.3663,  0.0000,  0.3663],
        [ 0.7326,  1.0989,  1.4652]])

Что, если для каждой строки матрицы у вас есть разные значения? В этом случае вам потребуется "распространить" вектор на матрицу.

#### Преобразование вектора в матрицу.


Мы можем преобразовать вектор в матрицу следующим образом:


In [ ]:
c = tensor([10.,20,30])
m = tensor([[1., 2, 3], [4,5,6], [7,8,9]])
m.shape,c.shape

(torch.Size([3, 3]), torch.Size([3]))

In [ ]:
m + c

tensor([[11., 22., 33.],
        [14., 25., 36.],
        [17., 28., 39.]])

Здесь элементы массива `c` расширяются, чтобы сформировать три строки, которые совпадают, что делает операцию возможной. Важно отметить, что PyTorch фактически не создает три копии массива `c` в памяти. Это достигается с помощью метода `expand_as`, который работает в фоновом режиме.

In [ ]:
c.expand_as(m)

tensor([[10., 20., 30.],
        [10., 20., 30.],
        [10., 20., 30.]])

Если мы посмотрим на соответствующий тензор, мы можем запросить его свойство `storage` (которое показывает фактическое содержимое памяти, используемой для этого тензора), чтобы убедиться, что в нем не хранится ненужная информация:


In [ ]:
t = c.expand_as(m)
t.storage()

 10.0
 20.0
 30.0
[torch.FloatStorage of size 3]

Хотя тензор формально имеет девять элементов, в памяти хранятся только три скалярных значения. Это возможно благодаря хитрому приему: указанию для этой размерности *шага* (stride) равного 0 (что означает, что при поиске следующей строки с использованием шага, PyTorch не перемещается).


In [ ]:
t.stride(), t.shape

((0, 1), torch.Size([3, 3]))

Поскольку `m` имеет размер 3x3, существует два способа выполнить широковещание (broadcasting). Тот факт, что оно было выполнено по последней размерности, является условностью, вытекающей из правил широковещания, и не имеет ничего общего с тем, как мы упорядочили наши тензоры. Если мы сделаем это другим способом, мы получим тот же результат:


In [ ]:
c + m

tensor([[11., 22., 33.],
        [14., 25., 36.],
        [17., 28., 39.]])

Фактически, передать вектор размера `n` можно только с помощью матрицы размера `m` x `n`:


In [ ]:
c = tensor([10.,20,30])
m = tensor([[1., 2, 3], [4,5,6]])
c+m

tensor([[11., 22., 33.],
        [14., 25., 36.]])

Это не сработает:


In [ ]:
c = tensor([10.,20])
m = tensor([[1., 2, 3], [4,5,6]])
c+m

RuntimeError: The size of tensor a (2) must match the size of tensor b (3) at non-singleton dimension 1

Если мы хотим передавать данные в другом измерении, нам нужно изменить форму нашего вектора, чтобы он стал матрицей размером 3x1. Это можно сделать с помощью метода `unsqueeze` в PyTorch:


In [ ]:
c = tensor([10.,20,30])
m = tensor([[1., 2, 3], [4,5,6], [7,8,9]])
c = c.unsqueeze(1)
m.shape,c.shape

(torch.Size([3, 3]), torch.Size([3, 1]))

В этот раз переменная `c` разворачивается по столбцам:


In [ ]:
c+m

tensor([[11., 12., 13.],
        [24., 25., 26.],
        [37., 38., 39.]])

Как и раньше, в памяти хранятся только три скалярных значения:


In [ ]:
t = c.expand_as(m)
t.storage()

 10.0
 20.0
 30.0
[torch.FloatStorage of size 3]

И расширенный тензор имеет правильную форму, потому что у размерности столбцов шаг равен 0:


In [ ]:
t.stride(), t.shape

((1, 0), torch.Size([3, 3]))

При передаче данных (broadcasting), по умолчанию, если необходимо добавить измерения, они добавляются в начале. Раньше, когда мы использовали broadcasting, PyTorch выполнял операцию `c.unsqueeze(0)` в фоновом режиме:

In [ ]:
c = tensor([10.,20,30])
c.shape, c.unsqueeze(0).shape,c.unsqueeze(1).shape

(torch.Size([3]), torch.Size([1, 3]), torch.Size([3, 1]))

Команда `unsqueeze` может быть заменена на указание `None` в качестве индекса:


In [ ]:
c.shape, c[None,:].shape,c[:,None].shape

(torch.Size([3]), torch.Size([1, 3]), torch.Size([3, 1]))

Вы всегда можете опускать двоеточия в конце, а `...` означает все предыдущие измерения:


In [ ]:
c[None].shape,c[...,None].shape

(torch.Size([1, 3]), torch.Size([3, 1]))

С помощью этого мы можем убрать еще один цикл `for` из нашей функции умножения матриц. Теперь, вместо умножения `a[i]` на `b[:,j]`, мы можем умножить `a[i]` на всю матрицу `b`, используя механизм широковещания (broadcasting), а затем суммировать результаты.

In [ ]:
def matmul(a,b):
    ar,ac = a.shape
    br,bc = b.shape
    assert ac==br
    c = torch.zeros(ar, bc)
    for i in range(ar):
#       c[i,j] = (a[i,:]          * b[:,j]).sum() # previous
        c[i]   = (a[i  ].unsqueeze(-1) * b).sum(dim=0)
    return c

In [ ]:
%timeit -n 20 t4 = matmul(m1,m2)

357 µs ± 7.2 µs per loop (mean ± std. dev. of 7 runs, 20 loops each)


Мы стали на 3700 раз быстрее, чем в нашей первой реализации! Прежде чем мы перейдем к следующей теме, давайте подробнее обсудим правила вещания.

#### Правила вещания


При работе с двумя тензорами, PyTorch сравнивает их формы поэлементно. Он начинает со *последних размерностей* и двигается в обратном направлении, добавляя 1, когда встречает пустые размерности. Две размерности считаются *совместимыми*, если выполняется одно из следующих условий:

- Они равны.
- Одна из них равна 1, в этом случае эта размерность "расширяется" (broadcast), чтобы сделать ее такой же, как другая.

Массивы не обязательно должны иметь одинаковое количество размерностей. Например, если у вас есть массив RGB значений размером 256x256x3, и вы хотите масштабировать каждый цвет в изображении на разные значения, вы можете умножить изображение на одномерный массив с тремя значениями. Сопоставление размеров последних осей этих массивов в соответствии с правилами "расширения" показывает, что они совместимы:

```
Изображение (тензор 3d): 256 x 256 x 3
Масштаб (тензор 1d): (1) (1) 3
Результат (тензор 3d): 256 x 256 x 3
```

Однако, двумерный тензор размером 256x256 не совместим с нашим изображением:

```
Изображение (тензор 3d): 256 x 256 x 3
Масштаб (тензор 2d): (1) 256 x 256
Ошибка
```

В предыдущих примерах, когда мы работали с матрицей 3x3 и вектором размера 3, "расширение" происходило по строкам:

```
Матрица (тензор 2d): 3 x 3
Вектор (тензор 1d): (1) 3
Результат (тензор 2d): 3 x 3
```

В качестве упражнения, попробуйте определить, какие размерности нужно добавить (и куда), когда вам нужно нормализовать пакет изображений размером `64 x 3 x 256 x 256` с использованием векторов из трех элементов (один для среднего значения и один для стандартного отклонения).


Еще один полезный способ упрощения операций с тензорами – это использование соглашения Эйнштейна о суммировании.


### Суммирование Эйнштейна


Перед использованием операции `@` или функции `torch.matmul` в PyTorch, существует еще один способ реализации умножения матриц: это суммирование Эйнштейна (`einsum`). Это компактное представление для объединения произведений и сумм в общем виде. Мы записываем уравнение следующим образом:

```
ik,kj -> ij
```

Левая часть представляет собой размерности операндов, разделенные запятыми. Здесь у нас два тензора, каждый из которых имеет две размерности (`i,k` и `k,j`). Правая часть представляет собой размерности результата, поэтому здесь у нас тензор с двумя размерностями `i,j`.

Правила обозначения суммирования Эйнштейна следующие:

1. Повторяющиеся индексы в левой части подразумевают суммирование по этим индексам, если они не указаны в правой части.
2. Каждый индекс может встречаться не более двух раз в левой части.
3. Неповторяющиеся индексы в левой части должны присутствовать в правой части.

Таким образом, в нашем примере, поскольку индекс `k` повторяется, мы суммируем по этому индексу. В итоге, формула представляет собой матрицу, полученную путем суммирования всех коэффициентов `(i,k)` в первом тензоре, умноженных на коэффициенты `(k,j)` во втором тензоре... что и есть произведение матриц! Вот как мы можем реализовать это в PyTorch:


In [ ]:
def matmul(a,b): return torch.einsum('ik,kj->ij', a, b)

Суммирование Эйнштейна – это очень удобный способ выражения операций, включающих индексацию и сумму произведений. Обратите внимание, что слева от знака равенства может быть только один член. Например:

```python
torch.einsum('ij->ji', a)
```

возвращает транспонированную матрицу `a`. Также может быть три или более члена. Например:

```python
torch.einsum('bi,ij,bj->b', a, b, c)
```

вернет вектор размера `b`, где `k`-я координата является суммой `a[k,i] * b[i,j] * c[k,j]`. Эта нотация особенно удобна, когда у вас есть больше измерений из-за использования пакетов (батчей). Например, если у вас есть два пакета матриц и вы хотите вычислить произведение матриц для каждого пакета, вы можете использовать:

```python
torch.einsum('bik,bkj->bij', a, b)
```

Теперь вернемся к нашей новой реализации `matmul` с использованием `einsum` и посмотрим на ее производительность:


In [ ]:
%timeit -n 20 t5 = matmul(m1,m2)

68.7 µs ± 4.06 µs per loop (mean ± std. dev. of 7 runs, 20 loops each)


Как вы видите, это не только практично, но и *очень* быстро. `einsum` часто является самым быстрым способом выполнения пользовательских операций в PyTorch, позволяющим избежать необходимости работы с C++ и CUDA. (Однако, как видно из результатов, представленных в разделе "Умножение матриц с нуля", он, как правило, не так быстр, как тщательно оптимизированный код CUDA).

Теперь, когда мы знаем, как реализовать умножение матриц с нуля, мы готовы построить нашу нейронную сеть, а именно ее прямой и обратный проходы, используя только операции умножения матриц.

## Прямой и обратный проходы.

Как мы видели в разделе <<chapter_mnist_basics>>, для обучения модели необходимо вычислить все градиенты заданной функции потерь по отношению к ее параметрам, что называется *обратным распространением ошибки* (backpropagation). *Прямое распространение* (forward pass) — это этап, на котором вычисляется выход модели для заданного входного значения, основываясь на матричных произведениях. При определении нашей первой нейронной сети мы также рассмотрим проблему правильной инициализации весов, что имеет решающее значение для успешного начала обучения.

### Определение и инициализация слоя


Давайте рассмотрим пример нейронной сети с двумя слоями. Как мы уже видели, один слой можно представить в виде `y = x @ w + b`, где `x` – входные данные, `y` – выходные данные, `w` – веса слоя (размерность которых равна количеству входных данных, умноженному на количество нейронов, если мы не транспонируем матрицу, как это было раньше), а `b` – вектор смещения.

In [ ]:
def lin(x, w, b): return x @ w + b

Мы можем разместить второй слой поверх первого, но, поскольку математически композиция двух линейных операций является еще одной линейной операцией, это имеет смысл только в том случае, если мы поместим нелинейный элемент между ними, который называется функцией активации. Как упоминалось в начале этой главы, в приложениях глубокого обучения наиболее часто используемой функцией активации является ReLU, которая возвращает максимум из `x` и `0`.

В этой главе мы фактически не будем обучать нашу модель, поэтому будем использовать случайные тензоры для входных данных и целевых значений. Предположим, что наши входные данные – это 200 векторов размера 100, которые мы объединяем в один пакет, а наши целевые значения – это 200 случайных чисел с плавающей точкой:


In [ ]:
x = torch.randn(200, 100)
y = torch.randn(200)

Для нашей двухслойной модели нам понадобятся две матрицы весов и два вектора смещений. Предположим, что размер скрытого слоя равен 50, а размер выходного слоя – 1 (для одного из наших входных данных соответствующим выходом является одно число с плавающей точкой в этом упрощенном примере). Мы инициализируем веса случайными значениями, а смещения – нулями:


In [ ]:
w1 = torch.randn(100,50)
b1 = torch.zeros(50)
w2 = torch.randn(50,1)
b2 = torch.zeros(1)

Затем результат работы нашего первого слоя будет следующим:


In [ ]:
l1 = lin(x, w1, b1)
l1.shape

torch.Size([200, 50])

Обратите внимание, что эта формула работает с нашим набором входных данных и возвращает набор скрытых состояний: `l1` – это матрица размером 200 (размер нашей партии) на 50 (размер нашего скрытого слоя).

Однако, существует проблема с тем, как была инициализирована наша модель. Чтобы понять ее, нам нужно посмотреть на среднее значение и стандартное отклонение (std) `l1`:


In [ ]:
l1.mean(), l1.std()

(tensor(0.0019), tensor(10.1058))

Среднее значение близко к нулю, что вполне логично, поскольку как входные данные, так и матрицы весов имеют средние значения, близкие к нулю. Однако стандартное отклонение, которое показывает, насколько сильно наши выходные значения отклоняются от среднего, увеличилось с 1 до 10. Это очень серьезная проблема, поскольку это касается только одного слоя. Современные нейронные сети могут иметь сотни слоев, поэтому, если каждый из них увеличивает масштаб наших выходных значений в 10 раз, к концу последнего слоя мы получим числа, которые компьютер не сможет представить.

Действительно, если мы выполним всего 50 умножений матрицы `x` на случайные матрицы размером 100x100, мы получим:


In [ ]:
x = torch.randn(200, 100)
for i in range(50): x = x @ torch.randn(100,100)
x[0:5,0:5]

tensor([[nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan]])

В результате мы получаем значения "не число" (NaN) повсюду. Возможно, масштаб нашей матрицы был слишком большим, и нам нужны меньшие веса? Но если мы используем слишком маленькие веса, у нас возникнет противоположная проблема: масштаб наших активаций изменится от 1 до 0.1, и после 50 слоев мы получим нули везде.

In [ ]:
x = torch.randn(200, 100)
for i in range(50): x = x @ (torch.randn(100,100) * 0.01)
x[0:5,0:5]

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

Таким образом, нам необходимо правильно подобрать масштабные коэффициенты для наших матриц весов, чтобы стандартное отклонение наших активаций оставалось равным 1. Точное значение для этого можно вычислить математически, как это показано в работе Ксавье Глоро и Йошуа Бенджио ["Understanding the Difficulty of Training Deep Feedforward Neural Networks"](http://proceedings.mlr.press/v9/glorot10a/glorot10a.pdf). Правильный масштаб для заданного слоя равен $1/\sqrt{n_{in}}$, где $n_{in}$ представляет собой количество входных данных.

В нашем случае, если у нас 100 входных данных, мы должны масштабировать наши матрицы весов коэффициентом 0,1:


In [ ]:
x = torch.randn(200, 100)
for i in range(50): x = x @ (torch.randn(100,100) * 0.1)
x[0:5,0:5]

tensor([[ 0.7554,  0.6167, -0.1757, -1.5662,  0.5644],
        [-0.1987,  0.6292,  0.3283, -1.1538,  0.5416],
        [ 0.6106,  0.2556, -0.0618, -0.9463,  0.4445],
        [ 0.4484,  0.7144,  0.1164, -0.8626,  0.4413],
        [ 0.3463,  0.5930,  0.3375, -0.9486,  0.5643]])

Наконец-то числа, которые не являются ни нулями, ни `NaN`! Обратите внимание на то, насколько стабильным остается масштаб наших активаций, даже после добавления этих 50 "ложных" слоев:


In [ ]:
x.std()

tensor(0.7042)

Если немного поэкспериментировать со значением параметра масштабирования, вы заметите, что даже небольшое отклонение от 0.1 приведет либо к очень маленьким, либо к очень большим числам, поэтому правильная инициализация весов имеет огромное значение.

Вернемся к нашей нейронной сети. Поскольку мы немного изменили входные данные, нам нужно их переопределить:


In [ ]:
x = torch.randn(200, 100)
y = torch.randn(200)

И для инициализации весов мы будем использовать правильный метод, который называется *Xavier initialization* (или *Glorot initialization*):

In [ ]:
from math import sqrt
w1 = torch.randn(100,50) / sqrt(100)
b1 = torch.zeros(50)
w2 = torch.randn(50,1) / sqrt(50)
b2 = torch.zeros(1)

Теперь, если мы вычислим результат работы первого слоя, мы можем проверить, находятся ли среднее значение и стандартное отклонение в пределах нормы:


In [ ]:
l1 = lin(x, w1, b1)
l1.mean(),l1.std()

(tensor(-0.0050), tensor(1.0000))

Очень хорошо. Теперь нам нужно применить функцию ReLU, давайте ее определим. Функция ReLU отбрасывает все отрицательные значения и заменяет их нулями, что можно также описать как "обрезание" нашего тензора до нуля:


In [ ]:
def relu(x): return x.clamp_min(0.)

Мы передаем наши сигналы активации через следующее устройство:


In [ ]:
l2 = relu(l1)
l2.mean(),l2.std()

(tensor(0.3961), tensor(0.5783))

И мы снова оказались в исходном положении: среднее значение наших активаций стало 0,4 (что вполне объяснимо, учитывая, что мы убрали отрицательные значения), а стандартное отклонение снизилось до 0,58. Как и раньше, после нескольких слоев, скорее всего, мы получим нули:


In [ ]:
x = torch.randn(200, 100)
for i in range(50): x = relu(x @ (torch.randn(100,100) * 0.1))
x[0:5,0:5]

tensor([[0.0000e+00, 1.9689e-08, 4.2820e-08, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 1.6701e-08, 4.3501e-08, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 1.0976e-08, 3.0411e-08, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 1.8457e-08, 4.9469e-08, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 1.9949e-08, 4.1643e-08, 0.0000e+00, 0.0000e+00]])

Это означает, что наша начальная настройка была выполнена неправильно. Почему? В то время, когда Glorot и Bengio написали свою статью, наиболее распространенной функцией активации в нейронных сетях была гиперболическая тангенс (tanh), которую они и использовали. Однако эта схема инициализации не учитывает наши функции ReLU. К счастью, кто-то другой уже провел необходимые расчеты и определил правильный масштаб, который нам следует использовать. В статье ["Delving Deep into Rectifiers: Surpassing Human-Level Performance"](https://arxiv.org/abs/1502.01852) (которую мы уже видели ранее — это статья, представившая ResNet), Kaiming He и его коллеги показали, что нам следует использовать следующий масштаб: $\sqrt{2 / n_{in}}$, где $n_{in}$ — количество входных данных нашей модели. Давайте посмотрим, что это нам дает:


In [ ]:
x = torch.randn(200, 100)
for i in range(50): x = relu(x @ (torch.randn(100,100) * sqrt(2/100)))
x[0:5,0:5]

tensor([[0.2871, 0.0000, 0.0000, 0.0000, 0.0026],
        [0.4546, 0.0000, 0.0000, 0.0000, 0.0015],
        [0.6178, 0.0000, 0.0000, 0.0180, 0.0079],
        [0.3333, 0.0000, 0.0000, 0.0545, 0.0000],
        [0.1940, 0.0000, 0.0000, 0.0000, 0.0096]])

Это лучше: в этот раз наши значения не все обнулены. Итак, давайте вернемся к определению нашей нейронной сети и используем эту процедуру инициализации (которая называется *инициализацией Кайминга* или *инициализацией Хэ*):


In [ ]:
x = torch.randn(200, 100)
y = torch.randn(200)

In [ ]:
w1 = torch.randn(100,50) * sqrt(2 / 100)
b1 = torch.zeros(50)
w2 = torch.randn(50,1) * sqrt(2 / 50)
b2 = torch.zeros(1)

Давайте рассмотрим масштаб наших активаций после прохождения первого линейного слоя и функции активации ReLU:


In [ ]:
l1 = lin(x, w1, b1)
l2 = relu(l1)
l2.mean(), l2.std()

(tensor(0.5661), tensor(0.8339))

Гораздо лучше! Теперь, когда наши веса правильно инициализированы, мы можем определить всю нашу модель:

In [ ]:
def model(x):
    l1 = lin(x, w1, b1)
    l2 = relu(l1)
    l3 = lin(l2, w2, b2)
    return l3

Это прямой проход. Теперь осталось только сравнить наш результат с метками, которые у нас есть (в данном примере, случайные числа), используя функцию потерь. В данном случае мы будем использовать среднеквадратичную ошибку. (Это упрощенная задача, и эта функция потерь является самой простой для дальнейших вычислений, а именно для вычисления градиентов).

Единственная особенность заключается в том, что наши выходные данные и целевые значения не имеют точно одинаковой структуры – после прохождения через модель мы получаем результат, похожий на этот:


In [ ]:
out = model(x)
out.shape

torch.Size([200, 1])

Чтобы избавиться от лишней размерности в конце массива, используем функцию `squeeze`:


In [ ]:
def mse(output, targ): return (output.squeeze(-1) - targ).pow(2).mean()

И теперь мы готовы вычислить нашу функцию потерь:


In [ ]:
loss = mse(out, y)

Вот и все, что касается прямого прохода. Теперь давайте рассмотрим градиенты.

### Градиенты и обратное распространение ошибки.


Мы видели, что PyTorch вычисляет все необходимые градиенты с помощью специального вызова `loss.backward`, но давайте рассмотрим, что происходит за кулисами.

Теперь наступает этап, когда нам нужно вычислить градиенты функции потерь по отношению ко всем весам нашей модели, то есть ко всем числам с плавающей точкой в `w1`, `b1`, `w2` и `b2`. Для этого нам потребуется немного математики, а именно *правило цепи*. Это правило дифференциального исчисления, которое определяет, как можно вычислить производную составной функции:

$$(g \circ f)'(x) = g'(f(x)) f'(x)$$


```
j: Я нахожу эту запись очень сложной для понимания, поэтому я предпочитаю думать об этом так: если `y = g(u)` и `u = f(x)`, то `dy/dx = dy/du * du/dx`. Обе записи означают одно и то же, поэтому используйте ту, которая вам больше подходит.
```

Наша функция потерь представляет собой сложную конструкцию, состоящую из различных компонентов: среднеквадратичной ошибки (которая, в свою очередь, является произведением среднего значения и степени двойки), второго линейного слоя, функции ReLU и первого линейного слоя. Например, если мы хотим получить градиенты функции потерь по отношению к параметру `b2`, и наша функция потерь определена следующим образом:

```
loss = mse(out, y) = mse(lin(l2, w2, b2), y)
```

Правило цепочки гласит, что у нас получается:
$$\frac{\text{d} loss}{\text{d} b_{2}} = \frac{\text{d} loss}{\text{d} out} \times \frac{\text{d} out}{\text{d} b_{2}} = \frac{\text{d}}{\text{d} out} mse(out, y) \times \frac{\text{d}}{\text{d} b_{2}} lin(l_{2}, w_{2}, b_{2})$$

Чтобы вычислить градиенты функции потерь по отношению к параметру $b_{2}$, нам сначала нужны градиенты функции потерь по отношению к нашему выходному значению `out`. То же самое справедливо, если мы хотим получить градиенты функции потерь по отношению к параметру $w_{2}$. Затем, чтобы получить градиенты функции потерь по отношению к параметрам $b_{1}$ или $w_{1}$, нам понадобятся градиенты функции потерь по отношению к параметру $l_{1}$, что, в свою очередь, требует градиентов функции потерь по отношению к параметру $l_{2}$, которые, в свою очередь, потребуют градиентов функции потерь по отношению к выходному значению `out`.

Таким образом, для вычисления всех необходимых градиентов для обновления параметров, нам нужно начинать с выходных данных модели и двигаться *в обратном направлении*, слой за слоем – именно поэтому этот этап известен как *обратное распространение ошибки*. Мы можем автоматизировать этот процесс, если каждая реализованная нами функция (`relu`, `mse`, `lin`) будет предоставлять свою функцию обратного распространения: то есть, как получить градиенты функции потерь по отношению к входным данным, исходя из градиентов функции потерь по отношению к выходным данным.

Здесь мы заполняем эти градиенты в атрибуте каждого тензора, подобно тому, как PyTorch делает это с помощью `.grad`.

Сначала мы получаем градиенты функции потерь по отношению к выходным данным нашей модели (которые являются входными данными для функции потерь). Мы "отменяем" операцию `squeeze`, которую выполнили в функции `mse`, а затем используем формулу, которая дает производную от $x^{2}$: $2x$. Производная от среднего значения равна просто $1/n$, где $n$ — это количество элементов во входных данных:


In [ ]:
def mse_grad(inp, targ): 
    # grad of loss with respect to output of previous layer
    inp.g = 2. * (inp.squeeze() - targ).unsqueeze(-1) / inp.shape[0]

Для градиентов функций ReLU и нашего линейного слоя мы используем градиенты функции потерь по отношению к выходу (в переменной `out.g`) и применяем правило цепочки для вычисления градиентов функции потерь по отношению к входу (в переменной `inp.g`). Правило цепочки говорит нам, что `inp.g = relu'(inp) * out.g`. Производная функции `relu` равна либо 0 (когда входные значения отрицательные), либо 1 (когда входные значения положительные), поэтому мы получаем:


In [ ]:
def relu_grad(inp, out):
    # grad of relu with respect to input activations
    inp.g = (inp>0).float() * out.g

Схема вычисления градиентов функции потерь относительно входных данных, весов и смещения в линейном слое остается той же:


In [ ]:
def lin_grad(inp, out, w, b):
    # grad of matmul with respect to input
    inp.g = out.g @ w.t()
    w.g = inp.t() @ out.g
    b.g = out.g.sum(0)

Мы не будем подробно останавливаться на математических формулах, которые их определяют, поскольку они не важны для наших целей, но если вам интересна эта тема, обязательно ознакомьтесь с отличными уроками по математическому анализу на платформе Khan Academy.

### Боковая панель: SymPy


SymPy — это библиотека для символьных вычислений, которая чрезвычайно полезна при работе с математическим анализом. Согласно [документации](https://docs.sympy.org/latest/tutorial/intro.html):


> Символьные вычисления занимаются вычислениями, выполняемыми с использованием символьных представлений математических объектов. Это означает, что математические объекты представляются точно, а не приближенно, и математические выражения, содержащие невычисленные переменные, остаются в символьной форме.

Для выполнения символьных вычислений, мы сначала определяем *символ*, а затем выполняем вычисления, следующим образом:


In [ ]:
from sympy import symbols,diff
sx,sy = symbols('sx sy')
diff(sx**2, sx)

2*sx

Здесь SymPy вычислила производную от `x**2` за нас! Она может вычислять производные от сложных выражений, упрощать и факторизовать уравнения, и многое другое. Сейчас практически нет причин для того, чтобы кто-либо занимался дифференциальным исчислением вручную: для вычисления градиентов PyTorch делает это за нас, а для отображения уравнений SymPy делает это за нас!

### Конец боковой панели


Как только мы определим эти функции, мы сможем использовать их для реализации обратного прохода. Поскольку градиент для каждого элемента автоматически вычисляется и помещается в соответствующий тензор, нам не нужно сохранять результаты работы этих функций `_grad` где-либо — нам просто нужно выполнить их в обратном порядке по сравнению с прямым проходом, чтобы гарантировать, что для каждой функции существует переменная `out.g`:


In [ ]:
def forward_and_backward(inp, targ):
    # forward pass:
    l1 = inp @ w1 + b1
    l2 = relu(l1)
    out = l2 @ w2 + b2
    # we don't actually need the loss in backward!
    loss = mse(out, targ)
    
    # backward pass:
    mse_grad(out, targ)
    lin_grad(l2, out, w2, b2)
    relu_grad(l1, l2)
    lin_grad(inp, l1, w1, b1)

Теперь мы можем получить доступ к градиентам параметров нашей модели, которые находятся в файлах `w1.g`, `b1.g`, `w2.g` и `b2.g`.

Мы успешно определили нашу модель – теперь давайте сделаем ее более похожей на модуль PyTorch.

### Рефакторинг модели.


В трех функциях, которые мы использовали, есть две связанные функции: прямой проход (forward pass) и обратный проход (backward pass). Вместо того, чтобы писать их отдельно, мы можем создать класс, который объединит их. Этот класс также может хранить входные и выходные данные для обратного прохода. Таким образом, нам нужно будет просто вызвать функцию `backward`:


In [ ]:
class Relu():
    def __call__(self, inp):
        self.inp = inp
        self.out = inp.clamp_min(0.)
        return self.out
    
    def backward(self): self.inp.g = (self.inp>0).float() * self.out.g

`__call__` – это специальное имя в Python, которое позволит нашему классу быть вызываемым. Именно этот метод будет выполнен, когда мы напишем `y = Relu()(x)`. Мы можем сделать то же самое для нашего линейного слоя и функции потерь MSE:


In [ ]:
class Lin():
    def __init__(self, w, b): self.w,self.b = w,b
        
    def __call__(self, inp):
        self.inp = inp
        self.out = inp@self.w + self.b
        return self.out
    
    def backward(self):
        self.inp.g = self.out.g @ self.w.t()
        self.w.g = self.inp.t() @ self.out.g
        self.b.g = self.out.g.sum(0)

In [ ]:
class Mse():
    def __call__(self, inp, targ):
        self.inp = inp
        self.targ = targ
        self.out = (inp.squeeze() - targ).pow(2).mean()
        return self.out
    
    def backward(self):
        x = (self.inp.squeeze()-self.targ).unsqueeze(-1)
        self.inp.g = 2.*x/self.targ.shape[0]

Затем мы можем поместить все это в модель, которую инициализируем с помощью наших тензоров `w1`, `b1`, `w2` и `b2`:


In [ ]:
class Model():
    def __init__(self, w1, b1, w2, b2):
        self.layers = [Lin(w1,b1), Relu(), Lin(w2,b2)]
        self.loss = Mse()
        
    def __call__(self, x, targ):
        for l in self.layers: x = l(x)
        return self.loss(x, targ)
    
    def backward(self):
        self.loss.backward()
        for l in reversed(self.layers): l.backward()

Что действительно хорошо в этом рефакторинге и организации различных элементов как слоев нашей модели, так это то, что теперь прямой и обратный проходы (forward и backward passes) стали очень простыми в реализации. Если мы хотим создать экземпляр нашей модели, нам просто нужно написать:


In [ ]:
model = Model(w1, b1, w2, b2)

Далее можно выполнить прямой проход, используя:


In [ ]:
loss = model(x, y)

И обратный проход с использованием:


In [ ]:
model.backward()

### Переход к PyTorch


Классы `Lin`, `Mse` и `Relu`, которые мы написали, имеют много общего, поэтому мы можем сделать так, чтобы все они наследовали от одного и того же базового класса:


In [ ]:
class LayerFunction():
    def __call__(self, *args):
        self.args = args
        self.out = self.forward(*args)
        return self.out
    
    def forward(self):  raise Exception('not implemented')
    def bwd(self):      raise Exception('not implemented')
    def backward(self): self.bwd(self.out, *self.args)

Затем нам нужно реализовать методы `forward` и `bwd` в каждом из наших подклассов:


In [ ]:
class Relu(LayerFunction):
    def forward(self, inp): return inp.clamp_min(0.)
    def bwd(self, out, inp): inp.g = (inp>0).float() * out.g

In [ ]:
class Lin(LayerFunction):
    def __init__(self, w, b): self.w,self.b = w,b
        
    def forward(self, inp): return inp@self.w + self.b
    
    def bwd(self, out, inp):
        inp.g = out.g @ self.w.t()
        self.w.g = inp.t() @ self.out.g
        self.b.g = out.g.sum(0)

In [ ]:
class Mse(LayerFunction):
    def forward (self, inp, targ): return (inp.squeeze() - targ).pow(2).mean()
    def bwd(self, out, inp, targ): 
        inp.g = 2*(inp.squeeze()-targ).unsqueeze(-1) / targ.shape[0]

Остальная часть нашей модели может быть такой же, как и раньше. Мы становимся все ближе и ближе к тому, как это реализовано в PyTorch. Каждая базовая функция, которую нам нужно дифференцировать, реализована как объект `torch.autograd.Function`, который имеет методы `forward` (прямой проход) и `backward` (обратный проход). PyTorch будет отслеживать все вычисления, которые мы выполняем, чтобы правильно выполнить обратный проход, если мы не установим атрибут `requires_grad` наших тензоров в значение `False`.

Написать такой объект (почти) так же просто, как написать наши исходные классы. Разница заключается в том, что мы выбираем, что сохранять, а что помещать в контекстную переменную (чтобы убедиться, что мы не сохраняем ничего лишнего), и мы возвращаем градиенты в методе `backward`. Очень редко возникает необходимость писать собственные объекты `Function`, но если вам когда-либо потребуется что-то нестандартное или вы хотите изменить градиенты обычной функции, вот как это можно сделать:


In [ ]:
from torch.autograd import Function

class MyRelu(Function):
    @staticmethod
    def forward(ctx, i):
        result = i.clamp_min(0.)
        ctx.save_for_backward(i)
        return result
    
    @staticmethod
    def backward(ctx, grad_output):
        i, = ctx.saved_tensors
        return grad_output * (i>0).float()

Структура, используемая для создания более сложной модели, которая использует указанные `Function` (функции), называется `torch.nn.Module`. Это базовая структура для всех моделей, и все нейронные сети, которые вы видели до сих пор, наследуют этот класс. Она в основном помогает регистрировать все обучаемые параметры, которые, как мы уже видели, могут использоваться в цикле обучения.

Для реализации `nn.Module` вам нужно:

- Убедиться, что метод `__init__` родительского класса вызывается в первую очередь при инициализации объекта.
- Определить все параметры модели как атрибуты с использованием `nn.Parameter`.
- Определить функцию `forward`, которая возвращает результат работы вашей модели.

В качестве примера, вот как можно реализовать линейный слой "с нуля":


In [ ]:
import torch.nn as nn

class LinearLayer(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_out, n_in) * sqrt(2/n_in))
        self.bias = nn.Parameter(torch.zeros(n_out))
    
    def forward(self, x): return x @ self.weight.t() + self.bias

Как вы видите, этот класс автоматически отслеживает, какие параметры были определены:


In [ ]:
lin = LinearLayer(10,2)
p1,p2 = lin.parameters()
p1.shape,p2.shape

(torch.Size([2, 10]), torch.Size([2]))

Благодаря этой особенности `nn.Module`, мы можем просто вызвать `opt.step()`, и оптимизатор сам пройдет по всем параметрам и обновит каждый из них.

Обратите внимание, что в PyTorch веса хранятся в виде матрицы `n_out x n_in`, поэтому в процессе прямого распространения сигнала используется транспонирование.

Используя линейный слой из PyTorch (который также использует инициализацию Кайминга), модель, которую мы строили в этой главе, может быть записана следующим образом:


In [ ]:
class Model(nn.Module):
    def __init__(self, n_in, nh, n_out):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_in,nh), nn.ReLU(), nn.Linear(nh,n_out))
        self.loss = mse
        
    def forward(self, x, targ): return self.loss(self.layers(x).squeeze(), targ)

`fastai` предоставляет свою собственную версию класса `Module`, которая идентична `nn.Module`, но не требует от вас вызова `super().__init__()` (она делает это автоматически).

In [ ]:
class Model(Module):
    def __init__(self, n_in, nh, n_out):
        self.layers = nn.Sequential(
            nn.Linear(n_in,nh), nn.ReLU(), nn.Linear(nh,n_out))
        self.loss = mse
        
    def forward(self, x, targ): return self.loss(self.layers(x).squeeze(), targ)

В последней главе мы начнем с определенной модели и посмотрим, как создать цикл обучения с нуля, а затем оптимизировать его, чтобы он соответствовал тому, что мы использовали в предыдущих главах.

## Заключение


В этой главе мы рассмотрели основы глубокого обучения, начиная с матричного умножения и переходя к реализации прямого и обратного проходов нейронной сети с нуля. Затем мы рефакторизовали наш код, чтобы показать, как работает PyTorch на более низком уровне.

Вот несколько важных моментов, которые стоит помнить:

- Нейронная сеть, по сути, представляет собой набор матричных умножений с нелинейностями между ними.
- Python работает медленно, поэтому для написания быстрого кода необходимо использовать векторизацию и такие техники, как арифметические операции над элементами и широковещание.
- Два тензора являются широковещательными (broadcastable), если их размерности, начиная с конца и двигаясь в обратном направлении, совпадают (если они одинаковы или одна из них равна 1). Чтобы сделать тензоры широковещательными, возможно, потребуется добавить размерности размера 1 с помощью функции `unsqueeze` или индекса `None`.
- Правильная инициализация нейронной сети имеет решающее значение для начала обучения. Инициализация Kaiming следует использовать, когда используются нелинейности ReLU.
- Обратный проход — это многократное применение правила цепочки, при котором вычисляются градиенты, начиная от выходных данных нашей модели и двигаясь обратно, слой за слоем.
- При создании подклассов `nn.Module` (если не используется `Module` из библиотеки fastai) необходимо вызывать метод `__init__` родительского класса в нашем методе `__init__`, и необходимо определить функцию `forward`, которая принимает входные данные и возвращает желаемый результат.

## Анкета


1. Напишите код на Python для реализации одного нейрона.
1. Напишите код на Python для реализации функции ReLU.
1. Напишите код на Python для реализации полносвязного слоя с использованием матричного умножения.
1. Напишите код на Python для реализации полносвязного слоя, используя обычный Python (то есть с использованием генераторов списков и встроенных функций Python).
1. Что такое "размер скрытого слоя"?
1. Что делает метод `t` в PyTorch?
2. Почему матричное умножение, написанное на обычном Python, работает очень медленно?
3. В функции `matmul`, почему `ac==br`?
4. Как измерить время выполнения отдельной ячейки в Jupyter Notebook?
5. Что такое "поэлементные арифметические операции"?
6. Напишите код на PyTorch, чтобы проверить, является ли каждый элемент массива `a` больше, чем соответствующий элемент массива `b`.
7. Что такое тензор ранга 0? Как его преобразовать в обычный тип данных Python?
8. Что возвращает и почему? `tensor([1,2]) + tensor([1])`
9. Что возвращает и почему? `tensor([1,2]) + tensor([1,2,3])`
10. Как поэлементные арифметические операции помогают ускорить `matmul`?
11. Какие правила вещания (broadcasting)?
12. Что такое `expand_as`? Приведите пример того, как его можно использовать для достижения результатов, аналогичных вещанию.
13. Как `unsqueeze` помогает решать определенные проблемы вещания?
14. Как можно использовать индексацию для выполнения той же операции, что и `unsqueeze`?
15. Как можно отобразить фактическое содержимое памяти, используемой для тензора?
16. При сложении вектора размера 3 с матрицей размера 3x3, добавляются ли элементы вектора к каждой строке или к каждому столбцу матрицы? (Обязательно проверьте свой ответ, запустив этот код в блокноте).
17. Приводят ли вещание и `expand_as` к увеличению использования памяти? Почему да или почему нет?
18. Реализуйте `matmul` с использованием суммирования Эйнштейна.
19. Что представляет собой повторяющаяся буква индекса в левой части выражения einsum?
20. Каковы три правила обозначения суммирования Эйнштейна? Почему?
21. Что такое прямой проход (forward pass) и обратный проход (backward pass) в нейронной сети?
22. Зачем нам сохранять некоторые значения активаций, вычисленные для промежуточных слоев, во время прямого прохода?
23. Какие недостатки возникают, если стандартное отклонение активаций сильно отличается от 1?
24. Как инициализация весов может помочь избежать этой проблемы?
25. Какова формула инициализации весов, чтобы получить стандартное отклонение, равное 1, для обычного линейного слоя и для линейного слоя, за которым следует ReLU?
26. Почему нам иногда приходится использовать метод `squeeze` в функциях потерь?
27. Что делает аргумент метода `squeeze`? Почему может быть важно включать этот аргумент, даже если PyTorch этого не требует?
28. Что такое "правило цепочки"? Приведите уравнение в одной из двух форм, представленных в этой главе.
29. Покажите, как вычислить градиенты `mse(lin(l2, w2, b2), y)` с использованием правила цепочки.
30. Каков градиент функции ReLU? Покажите его в виде математической формулы или кода. (Вам не нужно запоминать это — попробуйте вывести его, используя свои знания о форме функции).
31. В каком порядке нужно вызывать функции `*_grad` во время обратного прохода? Почему?
32. Что такое `__call__`?
33. Какие методы необходимо реализовать при создании `torch.autograd.Function`?
34. Напишите `nn.Linear` с нуля и протестируйте, работает ли он.
35. В чем разница между `nn.Module` и `Module` из библиотеки fastai?


### Дальнейшие исследования


1. Реализуйте функцию ReLU как `torch.autograd.Function` и обучите модель, используя ее.
2. Если вы хорошо разбираетесь в математике, выясните, какими являются градиенты линейного слоя, выраженные математическими формулами. Сопоставьте это с реализацией, которую мы рассмотрели в этой главе.
3. Изучите метод `unfold` в PyTorch и используйте его вместе с матричным умножением для реализации собственной функции 2D-свертки. Затем обучите сверточную нейронную сеть (CNN), которая использует ее.
4. Реализуйте все, что описано в этой главе, используя NumPy вместо PyTorch.